# Modelo de embeddings y clustering para robustecer el scoring de licitaciones

Este notebook está diseñado para ejecutarse **después** del notebook de scoring multifuente ya construido. Su objetivo no es reemplazar el score base, sino agregar dos componentes de Data Science:

1. **Modelo semántico con embeddings**: representa cada licitación como un vector numérico y calcula similitud semántica con el histórico de Los Tilos.
2. **Clustering**: agrupa licitaciones candidatas e históricas para analizar si las candidatas se ubican cerca de grupos documentales relacionados con Los Tilos.

El resultado final es un **score híbrido** que combina:

- score base interpretable;
- similitud semántica por embeddings;
- diagnóstico de agrupamiento mediante clustering.

> Nota metodológica: este modelo no estima probabilidad de adjudicación ni probabilidad de éxito. Es un modelo de recomendación y priorización documental basado en similitud.

## 0. Requisitos de ejecución

Este bloque espera que ya existan en memoria los siguientes objetos del notebook anterior:

- `df_candidatas`: licitaciones candidatas consolidadas.
- `df_tilos_hist`: histórico documental de Los Tilos.
- `score_df`: tabla con el score base ya calculado.

Si se ejecuta desde un notebook nuevo, primero debe correrse el notebook de scoring multifuente hasta terminar el cálculo de `score_df`.

In [1]:
# ============================================================
# 0. Validación de objetos requeridos
# ============================================================

required_objects = [
    "df_candidatas",
    "df_tilos_hist",
    "score_df",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Faltan objetos requeridos en memoria: "
        + ", ".join(missing_objects)
        + ". Ejecuta primero el notebook de scoring multifuente hasta calcular score_df."
    )

print("Objetos requeridos disponibles.")
print("df_candidatas:", df_candidatas.shape)
print("df_tilos_hist:", df_tilos_hist.shape)
print("score_df:", score_df.shape)

NameError: Faltan objetos requeridos en memoria: df_candidatas, df_tilos_hist, score_df. Ejecuta primero el notebook de scoring multifuente hasta calcular score_df.

## 1. Configuración del modelo

Se usa un modelo multilingüe de `sentence-transformers`, adecuado para textos en español. La primera ejecución puede tardar porque descarga el modelo. Después, el proceso se acelera mediante caché local de embeddings.

El modelo recomendado es:

`paraphrase-multilingual-MiniLM-L12-v2`

Es un modelo liviano, multilingüe y suficiente para un prototipo de TFM. Si el equipo quiere mayor precisión, puede sustituirse por un modelo más grande, con mayor costo computacional.

In [ ]:
# ============================================================
# 1. Imports y configuración
# ============================================================

import os
import re
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

# Ruta base del proyecto. Se asume ejecución desde Notebooks.
ruta_silver = Path("../data/Silver")
if not ruta_silver.exists():
    ruta_silver = Path("data/Silver")
if not ruta_silver.exists():
    ruta_silver = Path(".")

cache_dir = ruta_silver / "_cache_scoring"
fig_dir = ruta_silver / "figuras_scoring_modelos"
out_dir = ruta_silver / "outputs_scoring_modelos"

cache_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
FORZAR_RECALCULO_EMBEDDINGS = False
RANDOM_STATE = 42

print("ruta_silver:", ruta_silver.resolve())
print("cache_dir:", cache_dir.resolve())
print("fig_dir:", fig_dir.resolve())
print("out_dir:", out_dir.resolve())

## 2. Preparación del texto para embeddings

Para embeddings se usa el texto más completo disponible. El orden de prioridad es:

1. `texto_scoring`: consolidado de información estructurada, HTML y documentos.
2. `texto_actividad`: descripción enfocada en actividad contractual.
3. columnas alternativas si existieran.

Se aplica un límite de caracteres para evitar que documentos muy largos dominen el costo de procesamiento. Este límite no elimina la licitación; solo controla el tamaño del texto enviado al modelo.

In [ ]:
# ============================================================
# 2. Preparación del texto para embeddings
# ============================================================

MAX_CHARS_EMBEDDING = 12000


def normalize_text_for_embedding(value: object, max_chars: int = MAX_CHARS_EMBEDDING) -> str:
    """Limpia texto para el modelo de embeddings."""
    if pd.isna(value):
        return ""

    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()

    if len(text) > max_chars:
        text = text[:max_chars]

    return text


def build_embedding_text(df: pd.DataFrame) -> pd.Series:
    """Construye el texto que será vectorizado por embeddings."""
    candidate_cols = [
        "texto_scoring",
        "texto_actividad",
        "descripcion",
        "titulo",
        "objeto",
    ]

    existing_cols = [col for col in candidate_cols if col in df.columns]

    if not existing_cols:
        raise ValueError(
            "No se encontraron columnas textuales para construir embeddings."
        )

    text = (
        df[existing_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .map(normalize_text_for_embedding)
    )

    return text


df_candidatas = df_candidatas.copy()
df_tilos_hist = df_tilos_hist.copy()
score_df = score_df.copy()

df_candidatas["texto_embedding"] = build_embedding_text(df_candidatas)
df_tilos_hist["texto_embedding"] = build_embedding_text(df_tilos_hist)

print("Candidatas con texto embedding:", (df_candidatas["texto_embedding"].str.len() > 0).sum(), "de", len(df_candidatas))
print("Histórico Tilos con texto embedding:", (df_tilos_hist["texto_embedding"].str.len() > 0).sum(), "de", len(df_tilos_hist))

print("
Ejemplo texto candidata:")
print(df_candidatas["texto_embedding"].head(1).to_string(index=False)[:1500])

## 3. Carga del modelo de embeddings

Si `sentence-transformers` no está instalado, se debe instalar en el entorno:

```python
%pip install sentence-transformers
```

La instalación se deja comentada para no modificar automáticamente el entorno.

In [ ]:
# ============================================================
# 3. Carga del modelo de embeddings
# ============================================================

# Si hace falta instalar, descomentar esta línea:
# %pip install sentence-transformers

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError(
        "No está instalado sentence-transformers. Ejecuta: %pip install sentence-transformers"
    ) from exc

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Modelo cargado:", EMBEDDING_MODEL_NAME)

## 4. Generación y caché de embeddings

Los embeddings se guardan en archivos `.pkl` para no recalcularlos cada vez. La caché depende de:

- tipo de base: candidatas o histórico;
- número de registros;
- modelo usado.

Si se cambian textos de origen o se quiere recalcular todo, cambiar:

`FORZAR_RECALCULO_EMBEDDINGS = True`

In [ ]:
# ============================================================
# 4. Generación y caché de embeddings
# ============================================================


def get_cache_path(prefix: str, n_rows: int, model_name: str) -> Path:
    """Crea una ruta de caché para embeddings."""
    clean_model_name = re.sub(r"[^A-Za-z0-9_]+", "_", model_name)
    filename = f"{prefix}_{n_rows}_{clean_model_name}.pkl"
    return cache_dir / filename


def compute_or_load_embeddings(
    texts: list[str],
    prefix: str,
    model,
    model_name: str,
    force: bool = False,
) -> np.ndarray:
    """Carga embeddings desde caché o los calcula si no existen."""
    cache_path = get_cache_path(prefix, len(texts), model_name)

    if cache_path.exists() and not force:
        print(f"Cargando embeddings desde caché: {cache_path.name}")
        with open(cache_path, "rb") as file:
            embeddings = pickle.load(file)
        return embeddings

    print(f"Calculando embeddings para {prefix}: {len(texts)} textos")
    embeddings = model.encode(
        texts,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    with open(cache_path, "wb") as file:
        pickle.dump(embeddings, file)

    print(f"Embeddings guardados en caché: {cache_path.name}")
    return embeddings


candidate_texts = df_candidatas["texto_embedding"].fillna("").tolist()
history_texts = df_tilos_hist["texto_embedding"].fillna("").tolist()

emb_candidatas = compute_or_load_embeddings(
    candidate_texts,
    prefix="emb_candidatas",
    model=embedding_model,
    model_name=EMBEDDING_MODEL_NAME,
    force=FORZAR_RECALCULO_EMBEDDINGS,
)

emb_tilos = compute_or_load_embeddings(
    history_texts,
    prefix="emb_tilos",
    model=embedding_model,
    model_name=EMBEDDING_MODEL_NAME,
    force=FORZAR_RECALCULO_EMBEDDINGS,
)

print("Shape emb_candidatas:", emb_candidatas.shape)
print("Shape emb_tilos:", emb_tilos.shape)

## 5. Similitud semántica con el histórico de Los Tilos

Se calcula la similitud coseno entre cada licitación candidata y cada licitación histórica de Los Tilos.

Para cada candidata se generan tres indicadores:

- `score_embedding_top1`: máxima similitud contra una licitación histórica.
- `score_embedding_top3`: promedio de las tres mayores similitudes.
- `score_embedding_global`: similitud contra el perfil promedio de Los Tilos.

El `top3` es más estable que el `top1`, porque reduce el riesgo de que una sola coincidencia extrema domine el resultado.

In [ ]:
# ============================================================
# 5. Similitud semántica candidata vs histórico Los Tilos
# ============================================================

similarity_matrix = cosine_similarity(emb_candidatas, emb_tilos)

# IDs históricos disponibles.
if "id_licitacion_norm" in df_tilos_hist.columns:
    tilos_ids = df_tilos_hist["id_licitacion_norm"].astype(str).tolist()
else:
    tilos_ids = [f"tilos_{i}" for i in range(len(df_tilos_hist))]

# Top 1.
top1_idx = similarity_matrix.argmax(axis=1)
top1_score = similarity_matrix.max(axis=1)
top1_id = [tilos_ids[i] for i in top1_idx]

# Top 3.
top_k = min(3, similarity_matrix.shape[1])
top_k_idx = np.argsort(similarity_matrix, axis=1)[:, -top_k:][:, ::-1]
top_k_scores = np.take_along_axis(similarity_matrix, top_k_idx, axis=1)
top3_score = top_k_scores.mean(axis=1)
top3_ids = [
    ", ".join([tilos_ids[j] for j in row])
    for row in top_k_idx
]

# Perfil global de Los Tilos.
tilos_global_embedding = emb_tilos.mean(axis=0, keepdims=True)
tilos_global_embedding = tilos_global_embedding / np.linalg.norm(tilos_global_embedding)
global_score = cosine_similarity(emb_candidatas, tilos_global_embedding).ravel()

# Agregar a score_df. Se alinea por orden, asumiendo mismo orden que df_candidatas.
score_df["score_embedding_top1"] = top1_score
score_df["score_embedding_top3"] = top3_score
score_df["score_embedding_global"] = global_score
score_df["id_tilos_embedding_top1"] = top1_id
score_df["ids_tilos_embedding_top3"] = top3_ids

print("Resumen similitud embeddings:")
display(
    score_df[
        [
            "id_licitacion_norm",
            "score_embedding_top1",
            "score_embedding_top3",
            "score_embedding_global",
            "id_tilos_embedding_top1",
            "ids_tilos_embedding_top3",
        ]
    ]
    .sort_values("score_embedding_top3", ascending=False)
    .head(20)
)

## 6. Score híbrido: score base + embeddings

El score base es interpretable y auditable. El score de embeddings agrega similitud semántica.

Se propone un score híbrido simple:

\[
Score\_híbrido = 0.60 	imes Score\_embedding\_top3 + 0.40 	imes Score\_base
\]

La razón de usar 60 % embeddings es que este modelo captura similitud conceptual. Se mantiene 40 % del score base para conservar trazabilidad y reglas de negocio documentadas.

In [ ]:
# ============================================================
# 6. Score híbrido
# ============================================================


def find_base_score_column(df: pd.DataFrame) -> str:
    """Detecta la columna de score base en escala 0-100."""
    candidates = [
        "score_final_0_100",
        "score_total_0_100",
        "score_total_final_0_100",
        "score_hibrido_0_100",
    ]

    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        "No se encontró columna de score base en escala 0-100."
    )


base_score_col = find_base_score_column(score_df)
print("Columna de score base detectada:", base_score_col)

score_df["score_base_norm"] = score_df[base_score_col] / 100

score_df["score_hibrido_embedding"] = (
    0.60 * score_df["score_embedding_top3"]
    + 0.40 * score_df["score_base_norm"]
)

score_df["score_hibrido_embedding_0_100"] = (
    score_df["score_hibrido_embedding"] * 100
)


def classify_score(value: float) -> str:
    """Clasifica el score en niveles interpretables."""
    if pd.isna(value):
        return "Sin dato"
    if value < 40:
        return "Baja"
    if value < 60:
        return "Media"
    if value < 75:
        return "Alta"
    return "Muy alta"


score_df["nivel_hibrido_embedding"] = score_df[
    "score_hibrido_embedding_0_100"
].apply(classify_score)

cols_ranking_embedding = [
    "id_licitacion_norm",
    base_score_col,
    "score_embedding_top3",
    "score_hibrido_embedding_0_100",
    "nivel_hibrido_embedding",
    "id_tilos_embedding_top1",
]

if "nombre_licitacion" in score_df.columns:
    cols_ranking_embedding.insert(0, "nombre_licitacion")

print("Ranking híbrido con embeddings:")
display(
    score_df[cols_ranking_embedding]
    .sort_values("score_hibrido_embedding_0_100", ascending=False)
    .round(3)
    .head(20)
)

## 7. Comparación entre score base y score con embeddings

Esta comparación permite identificar si el modelo semántico mantiene, refuerza o cambia el ranking del scoring base.

Un cambio fuerte de posición no es automáticamente error; puede indicar que el modelo semántico detecta similitud conceptual no capturada por TF-IDF.

In [ ]:
# ============================================================
# 7. Comparación de rankings
# ============================================================

comparison_df = score_df.copy()
comparison_df["rank_base"] = comparison_df[base_score_col].rank(
    ascending=False,
    method="min",
)
comparison_df["rank_hibrido_embedding"] = comparison_df[
    "score_hibrido_embedding_0_100"
].rank(
    ascending=False,
    method="min",
)
comparison_df["cambio_rank"] = (
    comparison_df["rank_base"]
    - comparison_df["rank_hibrido_embedding"]
)

cols_comparison = [
    "id_licitacion_norm",
    base_score_col,
    "rank_base",
    "score_embedding_top3",
    "score_hibrido_embedding_0_100",
    "rank_hibrido_embedding",
    "cambio_rank",
]

if "nombre_licitacion" in comparison_df.columns:
    cols_comparison.insert(0, "nombre_licitacion")

print("Comparación ranking base vs ranking híbrido embeddings:")
display(
    comparison_df[cols_comparison]
    .sort_values("rank_hibrido_embedding")
    .round(3)
    .head(20)
)

print("
Correlación entre score base y score híbrido embeddings:")
print(
    comparison_df[[base_score_col, "score_hibrido_embedding_0_100"]]
    .corr()
    .round(3)
)

## 8. Clustering de licitaciones candidatas e históricas

El clustering se aplica sobre los embeddings combinados de:

- licitaciones candidatas;
- licitaciones históricas de Los Tilos.

El objetivo es identificar grupos temáticos. Si una candidata cae en un cluster con varios históricos de Los Tilos, esto refuerza la hipótesis de cercanía documental.

Se selecciona el número de clusters usando el índice de silueta en un rango razonable.

In [ ]:
# ============================================================
# 8. Selección de número de clusters
# ============================================================

combined_embeddings = np.vstack([emb_candidatas, emb_tilos])
combined_source = (
    ["candidata"] * len(df_candidatas)
    + ["tilos_hist"] * len(df_tilos_hist)
)

combined_ids = (
    score_df["id_licitacion_norm"].astype(str).tolist()
    + tilos_ids
)

max_k = min(10, len(combined_embeddings) - 1)
k_values = list(range(2, max_k + 1))

silhouette_results = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20,
    )
    labels = model.fit_predict(combined_embeddings)
    sil = silhouette_score(combined_embeddings, labels)
    silhouette_results.append({"k": k, "silhouette": sil})

silhouette_df = pd.DataFrame(silhouette_results)

best_k = int(
    silhouette_df.sort_values("silhouette", ascending=False)
    .iloc[0]["k"]
)

print("Resultados índice de silueta:")
display(silhouette_df.round(4))
print("Mejor k seleccionado:", best_k)

In [ ]:
# ============================================================
# 9. Entrenar clustering final
# ============================================================

cluster_model = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=20,
)

cluster_labels = cluster_model.fit_predict(combined_embeddings)

cluster_df = pd.DataFrame(
    {
        "id_licitacion_norm": combined_ids,
        "origen": combined_source,
        "cluster": cluster_labels,
    }
)

# Asignar clusters a candidatas.
score_df["cluster_embedding"] = cluster_labels[:len(df_candidatas)]

cluster_summary = (
    cluster_df
    .groupby(["cluster", "origen"])
    .size()
    .reset_index(name="n")
    .pivot_table(
        index="cluster",
        columns="origen",
        values="n",
        fill_value=0,
    )
    .reset_index()
)

for col in ["candidata", "tilos_hist"]:
    if col not in cluster_summary.columns:
        cluster_summary[col] = 0

cluster_summary["total"] = (
    cluster_summary["candidata"]
    + cluster_summary["tilos_hist"]
)

cluster_summary["pct_tilos_hist"] = (
    cluster_summary["tilos_hist"]
    / cluster_summary["total"]
).replace([np.inf, -np.inf], np.nan)

print("Resumen de clusters:")
display(cluster_summary.sort_values("cluster"))

print("Candidatas con cluster asignado:")
display(
    score_df[
        [
            "id_licitacion_norm",
            "cluster_embedding",
            "score_hibrido_embedding_0_100",
            "nivel_hibrido_embedding",
        ]
    ]
    .sort_values("score_hibrido_embedding_0_100", ascending=False)
    .head(20)
)

## 9. Interpretación de clusters con palabras clave

Para hacer interpretable el clustering, se extraen palabras representativas por cluster usando TF-IDF sobre el texto combinado. Esto no define el cluster, pero ayuda a nombrarlo.

In [ ]:
# ============================================================
# 10. Palabras clave por cluster
# ============================================================

combined_texts = candidate_texts + history_texts

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    ngram_range=(1, 2),
    stop_words=None,
)

tfidf_matrix = vectorizer.fit_transform(combined_texts)
feature_names = np.array(vectorizer.get_feature_names_out())

cluster_keywords = []

for cluster_id in sorted(cluster_df["cluster"].unique()):
    idx = np.where(cluster_labels == cluster_id)[0]
    if len(idx) == 0:
        continue

    mean_tfidf = np.asarray(tfidf_matrix[idx].mean(axis=0)).ravel()
    top_idx = mean_tfidf.argsort()[-10:][::-1]
    keywords = ", ".join(feature_names[top_idx])

    cluster_keywords.append(
        {
            "cluster": cluster_id,
            "keywords": keywords,
            "n_total": len(idx),
            "n_candidatas": int((cluster_df.loc[idx, "origen"] == "candidata").sum()),
            "n_tilos_hist": int((cluster_df.loc[idx, "origen"] == "tilos_hist").sum()),
        }
    )

cluster_keywords_df = pd.DataFrame(cluster_keywords)

print("Palabras clave por cluster:")
display(cluster_keywords_df)

# Unir palabras clave al score de candidatas.
score_df = score_df.merge(
    cluster_keywords_df[["cluster", "keywords"]],
    left_on="cluster_embedding",
    right_on="cluster",
    how="left",
)
score_df = score_df.drop(columns=["cluster"], errors="ignore")
score_df = score_df.rename(columns={"keywords": "keywords_cluster"})

## 10. Visualización PCA de clusters

Se proyectan los embeddings a dos componentes mediante PCA. Esta visualización no reemplaza el modelo, pero permite revisar si las candidatas se ubican cerca de regiones con históricos de Los Tilos.

In [ ]:
# ============================================================
# 11. Visualización 2D con PCA
# ============================================================

pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(combined_embeddings)

plot_cluster_df = cluster_df.copy()
plot_cluster_df["pc1"] = coords[:, 0]
plot_cluster_df["pc2"] = coords[:, 1]

plt.figure(figsize=(10, 7))

for origin in ["tilos_hist", "candidata"]:
    subset = plot_cluster_df[plot_cluster_df["origen"] == origin]
    plt.scatter(
        subset["pc1"],
        subset["pc2"],
        label=origin,
        alpha=0.75,
        s=70 if origin == "candidata" else 35,
    )

plt.title("Proyección PCA de embeddings por origen")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend()
plt.tight_layout()

pca_path = fig_dir / "pca_embeddings_origen.png"
plt.savefig(pca_path, dpi=160, bbox_inches="tight")
plt.show()

print("Figura guardada en:", pca_path)

In [ ]:
# ============================================================
# 12. Ranking híbrido final con información de clusters
# ============================================================

cols_final_modelo = [
    "id_licitacion_norm",
    base_score_col,
    "score_embedding_top3",
    "score_hibrido_embedding_0_100",
    "nivel_hibrido_embedding",
    "cluster_embedding",
    "keywords_cluster",
    "id_tilos_embedding_top1",
    "ids_tilos_embedding_top3",
]

if "nombre_licitacion" in score_df.columns:
    cols_final_modelo.insert(0, "nombre_licitacion")

cols_final_modelo = [
    col for col in cols_final_modelo
    if col in score_df.columns
]

ranking_modelo_df = (
    score_df[cols_final_modelo]
    .sort_values("score_hibrido_embedding_0_100", ascending=False)
    .reset_index(drop=True)
)

numeric_cols = ranking_modelo_df.select_dtypes(include="number").columns
ranking_modelo_df[numeric_cols] = ranking_modelo_df[numeric_cols].round(3)

print("Ranking final robustecido con embeddings y clustering:")
display(ranking_modelo_df.head(20))

## 11. Gráficas finales del modelo semántico

Estas gráficas sirven para presentar el aporte del modelo de embeddings y la comparación frente al score base.

In [ ]:
# ============================================================
# 13. Gráficas finales
# ============================================================

# ---------- Gráfica 1: score base vs híbrido embeddings ----------
plot_df = comparison_df.copy()

plt.figure(figsize=(8, 6))
plt.scatter(
    plot_df[base_score_col],
    plot_df["score_hibrido_embedding_0_100"],
    alpha=0.8,
    s=80,
)

min_val = min(plot_df[base_score_col].min(), plot_df["score_hibrido_embedding_0_100"].min())
max_val = max(plot_df[base_score_col].max(), plot_df["score_hibrido_embedding_0_100"].max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.title("Comparación: score base vs score híbrido con embeddings")
plt.xlabel("Score base (0-100)")
plt.ylabel("Score híbrido embeddings (0-100)")
plt.tight_layout()

fig_path = fig_dir / "score_base_vs_hibrido_embeddings.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()
print("Figura guardada en:", fig_path)

# ---------- Gráfica 2: top 10 por score híbrido ----------
if "nombre_licitacion" in score_df.columns:
    label_col = "nombre_licitacion"
else:
    label_col = "id_licitacion_norm"

top_plot = (
    score_df
    .sort_values("score_hibrido_embedding_0_100", ascending=False)
    .head(10)
    .copy()
)

top_plot["label_plot"] = top_plot[label_col].astype(str).str.slice(0, 55)
top_plot = top_plot.sort_values("score_hibrido_embedding_0_100", ascending=True)

plt.figure(figsize=(11, 7))
bars = plt.barh(
    top_plot["label_plot"],
    top_plot["score_hibrido_embedding_0_100"],
)

for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}",
        va="center",
        fontsize=9,
    )

plt.title("Top 10 licitaciones por score híbrido con embeddings")
plt.xlabel("Score híbrido embeddings (0-100)")
plt.ylabel("Licitación")
plt.xlim(0, max(top_plot["score_hibrido_embedding_0_100"]) + 10)
plt.tight_layout()

fig_path = fig_dir / "top10_score_hibrido_embeddings.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()
print("Figura guardada en:", fig_path)

# ---------- Gráfica 3: distribución por nivel híbrido ----------
nivel_counts = (
    score_df["nivel_hibrido_embedding"]
    .value_counts()
    .reindex(["Baja", "Media", "Alta", "Muy alta"], fill_value=0)
)

plt.figure(figsize=(8, 5))
bars = plt.bar(nivel_counts.index, nivel_counts.values)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.1,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.title("Distribución de niveles — score híbrido embeddings")
plt.xlabel("Nivel de compatibilidad")
plt.ylabel("Número de licitaciones")
plt.tight_layout()

fig_path = fig_dir / "niveles_score_hibrido_embeddings.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()
print("Figura guardada en:", fig_path)

## 12. Exportación de resultados

Se exportan las tablas principales para usarlas en la memoria del TFM o en Power BI/Excel.

In [ ]:
# ============================================================
# 14. Exportación de resultados
# ============================================================

output_excel = out_dir / "scoring_embeddings_clustering_resultados.xlsx"
output_parquet = out_dir / "scoring_embeddings_clustering_resultados.parquet"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    ranking_modelo_df.to_excel(writer, sheet_name="ranking_hibrido", index=False)
    comparison_df[cols_comparison].to_excel(writer, sheet_name="comparacion_scores", index=False)
    cluster_summary.to_excel(writer, sheet_name="resumen_clusters", index=False)
    cluster_keywords_df.to_excel(writer, sheet_name="keywords_clusters", index=False)
    silhouette_df.to_excel(writer, sheet_name="silhouette", index=False)

score_df.to_parquet(output_parquet, index=False)

print("Excel exportado:", output_excel.resolve())
print("Parquet exportado:", output_parquet.resolve())

## 13. Interpretación metodológica para el TFM

Texto sugerido:

> Como complemento al scoring documental base, se incorporó un modelo semántico basado en embeddings multilingües. Este modelo transforma cada licitación en un vector numérico y permite calcular la similitud coseno entre licitaciones candidatas y el histórico documental de Los Tilos. A diferencia del enfoque TF-IDF, que depende principalmente de coincidencias léxicas, los embeddings permiten capturar relaciones semánticas entre textos con vocabulario diferente pero significado cercano.
>
> Además, se aplicó clustering sobre los embeddings combinados de licitaciones candidatas e históricas. El objetivo fue identificar agrupamientos temáticos y verificar si las licitaciones candidatas se ubican en grupos donde también existen licitaciones históricas de Los Tilos. Este análisis no reemplaza el score final, sino que actúa como diagnóstico adicional para interpretar la cercanía documental.
>
> Finalmente, se calculó un score híbrido que combina el score base interpretable con la similitud semántica por embeddings. Esta estrategia conserva la trazabilidad del scoring inicial y agrega capacidad semántica al modelo de recomendación.